# HadISD Data Download Notebook

This notebook will help you download a subset of the HadISD dataset directly from the Met Office website. The data will be stored in a user-specified directory (or a sensible default), and extracted for further processing (e.g., conversion to Zarr).

- **Source:** [HadISD v3.4.0.2023f WMO_000000-029999.tar.gz](https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/data/WMO_000000-029999.tar.gz)
- **Instructions:**
    1. Set the download directory (or use the default).
    2. Download the data using Python's `requests` (recommended for stability) or `wget`.
    3. Extract the `.tar.gz` archive.
    4. The extracted files will be ready for use in the next notebook (conversion to Zarr).

> **Note:** Download size is large. Ensure you have sufficient disk space and a stable internet connection.

In [ ]:
import os
from pathlib import Path

# Set the download directory (user can change this if desired)
default_dir = Path.home() / "HadISD_data"
download_dir = os.environ.get("HADISD_DOWNLOAD_DIR", str(default_dir))
download_dir = Path(download_dir)
download_dir.mkdir(parents=True, exist_ok=True)

print(f"Data will be downloaded to: {download_dir}")

In [ ]:
import requests
from tqdm.auto import tqdm

url = "https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/data/WMO_080000-099999.tar.gz"
filename = download_dir / "WMO_080000-099999.tar"

# Download with resume support
headers = {}
if filename.exists():
    headers['Range'] = f'bytes={filename.stat().st_size}-'
    mode = 'ab'
else:
    mode = 'wb'

response = requests.get(url, stream=True, headers=headers)
total = int(response.headers.get('content-length', 0)) + (filename.stat().st_size if filename.exists() else 0)

with open(filename, mode) as f, tqdm(
    desc=f"Downloading {filename.name}",
    total=total,
    unit='B', unit_scale=True, unit_divisor=1024
) as bar:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)
            bar.update(len(chunk))

print(f"Download complete: {filename}")

In [ ]:
import tarfile

# Extract the tar.gz file
extract_dir = download_dir / "WMO_080000-099999"
extract_dir.mkdir(exist_ok=True)

with tarfile.open(filename, "r:gz") as tar:
    tar.extractall(path=extract_dir)

print(f"Extraction complete. Files are in: {extract_dir}")

In [ ]:
import gzip
import shutil

# --- Create subfolder for netcdf ---
netcdf_dir = download_dir / "WMO_080000-099999" / "netcdf"
netcdf_dir.mkdir(parents=True, exist_ok=True)

# Move extracted .nc files into netcdf_dir after extraction
for gz_path in extract_dir.glob('*.nc.gz'):
    nc_path = gz_path.with_suffix('')  # Remove .gz extension
    with gzip.open(gz_path, 'rb') as f_in, open(nc_path, 'wb') as f_out:
        f_out.write(f_in.read())
    print(f"Extracted: {nc_path}")
    gz_path.unlink()  # Delete the .gz file after extraction
    print(f"Deleted: {gz_path}")
    # Move the .nc file to netcdf_dir
    shutil.move(str(nc_path), netcdf_dir / nc_path.name)
    print(f"Moved: {nc_path} -> {netcdf_dir / nc_path.name}")

print("All .nc.gz files have been extracted, cleaned up, and moved to the netcdf directory.")